# Datavidia 11.0 shipment classification
Select **Runtime → Change runtime type → GPU** and run the setup cells in order. KaggleHub downloads the competition files into the Colab runtime. Google Drive stores the editable configuration and run artifacts.

In [ ]:
from pathlib import Path
import base64
import os
import re
import subprocess
import sys

REPO = "keizandrea/arkavidia"
BRANCH = "main"
PRIVATE_REPO = True

if REPO.startswith("YOUR_") or not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", REPO):
    raise ValueError("Set REPO to your GitHub owner and repository name.")

env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"
if PRIVATE_REPO:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    encoded = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
               GIT_CONFIG_VALUE_0=f"Authorization: Basic {encoded}")
    del token, encoded

repo_url = f"https://github.com/{REPO}.git"
project = Path("/content/arkavidia")
if project.exists():
    remote = subprocess.check_output(
        ["git", "-C", str(project), "remote", "get-url", "origin"], text=True
    ).strip()
    if remote != repo_url:
        raise ValueError("Existing runtime checkout points to a different repository. Start a fresh runtime.")
    subprocess.run(["git", "-C", str(project), "pull", "--ff-only", "origin", BRANCH], env=env, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, repo_url, str(project)],
                   env=env, check=True)
del env
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=project, check=True)

For a private repository, create a GitHub fine-grained token scoped to that repository with **Contents: Read-only**. Add it under **Secrets** in Colab as `GITHUB_TOKEN` and turn on notebook access. GitHub's notebook picker opens the notebook. The token separately lets the runtime clone the project code.

Join the Datavidia competition on Kaggle and accept its rules. Add a Kaggle API token to Colab **Secrets** under `KAGGLE_API_TOKEN` and enable notebook access. KaggleHub reads this secret directly.

In [ ]:
import json
import shutil

from google.colab import drive

subprocess.run([sys.executable, "-m", "pip", "install", "kagglehub>=1.0,<2"], check=True)
import kagglehub

data = Path(kagglehub.competition_download("task-1-datavidia-11-0")).resolve()
print("Path to competition files:", data)

drive.mount("/content/drive")
storage = Path("/content/drive/MyDrive/arkavidia")
config = storage / "config" / "colab.toml"
config.parent.mkdir(parents=True, exist_ok=True)
if not config.exists():
    shutil.copyfile(project / "config" / "default.toml", config)
print("Edit this TOML after inspecting the files:", config)

runtime_config = Path("/content/arkavidia-kaggle.toml")

def pipeline(*args):
    # Re-read the Drive config on each call and override only its data root.
    source = config.read_text(encoding="utf-8")
    data_section = re.search(r"(?m)^\s*\[data\][ \t]*\n(?P<body>(?:(?!^[ \t]*\[)[\s\S])*)", source)
    if data_section is None:
        raise ValueError("Expected a [data] section in the Drive config.")
    start, end = data_section.span("body")
    body, replaced = re.subn(
        r"(?m)^[ \t]*root[ \t]*=.*$", lambda _: "root = " + json.dumps(str(data), ensure_ascii=False),
        source[start:end],
    )
    if replaced != 1:
        raise ValueError("Expected exactly one root setting under [data].")
    runtime_config.write_text(source[:start] + body + source[end:], encoding="utf-8")
    subprocess.run([sys.executable, "run.py", "--config", str(runtime_config), *args], cwd=project, check=True)

pipeline("inspect")

Use the inspection output to edit `MyDrive/arkavidia/config/colab.toml`. Set the train, test, and sample filenames relative to the downloaded folder. Set the ID, target, prediction-time features, and auxiliary tables. The notebook supplies the download location for each run. See the project README for relation and date examples.

In [ ]:
from datetime import datetime, timezone

subprocess.run(["nvidia-smi", "-L"], check=True)
run_root = storage / "outputs"
run_root.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run = run_root / run_id
pipeline("train", "--profile", "baseline", "--device", "gpu", "--out", str(run))
print("Submission:", run / "submission.csv")
print("Metrics:", run / "metrics.json")

In [ ]:
from google.colab import files
files.download(str(run / "submission.csv"))